Data Cleaning in Pandas

In [69]:
import pandas as pd

In [71]:
# Team Goalie Stats refers to a collection of statistics specifically related to the performance 
# of each goalkeepers (goalies) on a team during a game or over multiple games.
# understand the dataset

df_game_goalie_stats = pd.read_csv(r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\extract\raw_data\game_goalie_stats.csv")
df_game_goalie_stats.info()
df_game_goalie_stats.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56656 entries, 0 to 56655
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   game_id                     56656 non-null  int64  
 1   player_id                   56656 non-null  int64  
 2   team_id                     56656 non-null  int64  
 3   timeOnIce                   56656 non-null  int64  
 4   assists                     56656 non-null  int64  
 5   goals                       56656 non-null  int64  
 6   pim                         56656 non-null  int64  
 7   shots                       56656 non-null  int64  
 8   saves                       56656 non-null  int64  
 9   powerPlaySaves              56656 non-null  int64  
 10  shortHandedSaves            56656 non-null  int64  
 11  evenSaves                   56656 non-null  int64  
 12  shortHandedShotsAgainst     56656 non-null  int64  
 13  evenShotsAgainst            566

,game_id,player_id,team_id,timeOnIce,assists,goals,pim,shots,saves,powerPlaySaves,shortHandedSaves,evenSaves,shortHandedShotsAgainst,evenShotsAgainst,powerPlayShotsAgainst,decision,savePercentage,powerPlaySavePercentage,evenStrengthSavePercentage
0,2016020045,8473607,4,1504,0,0,0,16,12,1,0,11,0,13,3,NaN,75.000000,33.333333,84.615385
1,2016020045,8473461,4,2011,0,0,0,11,9,1,0,8,0,10,1,L,81.818182,100.000000,80.000000
2,2016020045,8470645,16,3598,0,0,0,27,23,2,0,21,0,23,4,W,85.185185,50.000000,91.304348
3,2017020812,8468011,24,3696,0,0,0,33,30,1,2,27,3,28,2,W,90.909091,50.000000,96.428571
4,2017020812,8475215,7,3539,0,0,0,33,29,4,1,24,1,27,5,L,87.878788,80.000000,88.888889


In [73]:
# Create a new dataframe from df_game_goals (copying original to preserve data)
df_clean_game_goalie_stats = df_game_goalie_stats.copy()

Rename columns 

In [76]:
#Rename columns for consistency

import re

# Function to add an underscore before uppercase letters and convert to lowercase
def rename_columns(col_name):
    return re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', col_name).lower()

# Apply the function to all column names
df_clean_game_goalie_stats.columns = [rename_columns(col) for col in df_clean_game_goalie_stats.columns]
df_clean_game_goalie_stats.info()

# To ensure data consistency with df - game_team_stats and game_skater_stats, rename pim column
df_clean_game_goalie_stats.rename(columns={'pim': 'penalty_minutes'}, inplace=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56656 entries, 0 to 56655
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   game_id                        56656 non-null  int64  
 1   player_id                      56656 non-null  int64  
 2   team_id                        56656 non-null  int64  
 3   time_on_ice                    56656 non-null  int64  
 4   assists                        56656 non-null  int64  
 5   goals                          56656 non-null  int64  
 6   pim                            56656 non-null  int64  
 7   shots                          56656 non-null  int64  
 8   saves                          56656 non-null  int64  
 9   power_play_saves               56656 non-null  int64  
 10  short_handed_saves             56656 non-null  int64  
 11  even_saves                     56656 non-null  int64  
 12  short_handed_shots_against     56656 non-null 

Check : Null 

In [79]:
#Null Values : decision, save_percentage_power_play_save_percentage, even_strength_save_percentage
df_clean_game_goalie_stats.isnull().sum()

game_id                             0
player_id                           0
team_id                             0
time_on_ice                         0
assists                             0
goals                               0
penalty_minutes                     0
shots                               0
saves                               0
power_play_saves                    0
short_handed_saves                  0
even_saves                          0
short_handed_shots_against          0
even_shots_against                  0
power_play_shots_against            0
decision                         4102
save_percentage                   139
power_play_save_percentage       4743
even_strength_save_percentage     197
dtype: int64

In [83]:
# Handling Null Values in the decision Column of the game_goalie_stats DataFrame
# Action: To address null values in the decision column of the game_goalie_stats DataFrame, the merge and apply methods 
# in Python were used. The 'home_team_id', 'away_team_id', 'hoa' column from the game.csv was utilized to determine the 
# appropriate values for the decision column in game_goalie_stats.csv.
# Logic Applied:
# When hoa is 'home' and the team_id matches the home_team_id, the decision is set to 'W' (Win).
# When hoa is 'away' and the team_id matches the home_team_id, the decision is set to 'L' (Loss).


import pandas as pd

# Load the data
df_clean_game = pd.read_csv(r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\clean\game.csv")

merged_df = pd.merge(df_clean_game_goalie_stats, df_clean_game[['game_id', 'home_team_id', 'away_team_id', 'hoa']], 
                     on='game_id', how='left')

merged_df['decision'] = merged_df.apply(
    lambda row: row['decision'] if pd.isnull(row['hoa']) else row['decision'], axis=1
)

# Assign 'W' or 'L' based on whether the team is the home or away team
merged_df['decision'] = merged_df.apply(
    lambda row: 'W' if row['hoa'] == 'home' and row['team_id'] == row['home_team_id'] else row['decision'], axis=1
)

merged_df['decision'] = merged_df.apply(
    lambda row: 'L' if row['hoa'] == 'away' and row['team_id'] == row['home_team_id'] else row['decision'], axis=1
)

merged_df['decision'] = merged_df.apply(
    lambda row: 'W' if row['hoa'] == 'away' and row['team_id'] == row['away_team_id'] else row['decision'], axis=1
)

merged_df['decision'] = merged_df.apply(
    lambda row: 'L' if row['hoa'] == 'home' and row['team_id'] == row['away_team_id'] else row['decision'], axis=1
)

In [85]:
#drop the merged columns from game.csv 
df_clean_game_goalie_stats = merged_df.drop(columns=['home_team_id', 'away_team_id', 'hoa'])
df_clean_game_goalie_stats.isnull().sum()

#Check Results : decision column has 38 values.
#Since the null values for 'decision' is trivia, we will drop the null values.
df_clean_game_goalie_stats = df_clean_game_goalie_stats.dropna(subset=['decision'])
df_clean_game_goalie_stats.isnull().sum()

df_clean_game_goalie_stats.isnull().sum()

game_id                             0
player_id                           0
team_id                             0
time_on_ice                         0
assists                             0
goals                               0
penalty_minutes                     0
shots                               0
saves                               0
power_play_saves                    0
short_handed_saves                  0
even_saves                          0
short_handed_shots_against          0
even_shots_against                  0
power_play_shots_against            0
decision                            0
save_percentage                   133
power_play_save_percentage       4725
even_strength_save_percentage     191
dtype: int64

In [87]:
# Null values in 'save_percentage', 'power_play_save_percentage' and 'even_strength_play_percentage' column in game_goalie_stats table. [data is incomplete but not erroneous]
# Consideration:
# The formula for calculating save_percentage is save / shots * 100. If the saves are 0, the value should be 0% instead of NULL, 
# reflecting poor performance. This indicates no shots were saved.

# Consideration:
# When both saves and shots are 0, the save_percentage becomes undefined. In such cases, it is recommended to exclude these records 
# from calculations (such as averages or totals), as they are likely to skew results.

# Challenge: A "division by zero" error arises when the shots value is 0. To handle this, we use a conditional check to ensure that 
# division is only performed when shots is non-zero.

# Results: The recalculations for save_percentage are correct. However, there are 139 records with NULL values in save_percentage. 
# Recommend : Retain the NULL value for later analysis, to reflect the lack of data or shots faced, acknowledging the nature of the dataset.

In [89]:
# Step 1: Handle save_percentage NULL values
def calculate_save_percentage(row):
    if row['shots'] == 0:
        return None  # or you can return 0 if you want to treat 0 shots as perfect
    return round((row['saves'] / row['shots']) * 100, 2)

# Calculate the recalculated save percentage
df_clean_game_goalie_stats['recalculated_save_percentage'] = df_clean_game_goalie_stats.apply(calculate_save_percentage, axis=1)

# Step 2: Integrity Check - Compare stored and recalculated save_percentage
def check_integrity(row):
    if row['shots'] == 0:
        return 'Correct' if pd.isnull(row['save_percentage']) else 'Incorrect'
    return 'Correct' if round(row['save_percentage'], 2) == row['recalculated_save_percentage'] else 'Incorrect'

df_clean_game_goalie_stats['integrity_check'] = df_clean_game_goalie_stats.apply(check_integrity, axis=1)

# Step 3: Identify rows where save_percentage is NULL
null_save_percentage_rows = df_clean_game_goalie_stats[df_clean_game_goalie_stats['save_percentage'].isnull()]

# Results
print(df_clean_game_goalie_stats[['game_id', 'player_id', 'team_id', 'shots', 'saves', 'save_percentage', 'recalculated_save_percentage', 'integrity_check']])
print(f"\nRows with NULL save_percentage: \n{null_save_percentage_rows}")

# Count the number of rows where integrity check is not correct
incorrect_integrity_count = df_clean_game_goalie_stats[df_clean_game_goalie_stats['integrity_check'] == 'Incorrect'].shape[0]

# Output the result
print(f"Number of rows with integrity check not correct: {incorrect_integrity_count}")


          game_id  player_id  team_id  shots  saves  save_percentage  \
0      2016020045    8473607        4     16     12        75.000000   
1      2016020045    8473461        4     11      9        81.818182   
2      2016020045    8470645       16     27     23        85.185185   
3      2017020812    8468011       24     33     30        90.909091   
4      2017020812    8475215        7     33     29        87.878788   
...           ...        ...      ...    ...    ...              ...   
56651  2018030416    8476412       19     31     27        87.096774   
56652  2018030417    8476412       19     33     32        96.969697   
56653  2018030417    8471695        6     20     16        80.000000   
56654  2018030417    8476412       19     33     32        96.969697   
56655  2018030417    8471695        6     20     16        80.000000   

       recalculated_save_percentage integrity_check  
0                             75.00         Correct  
1                          

In [90]:
# Check integrity of power_play_save_percentage column
# Step 1: Handle power_play_save_percentage NULL values
def calculate_power_play_save_percentage(row):
    if row['power_play_shots_against'] == 0:
        return None  # or you can return 100 if you want to treat 0 shots as perfect
    return round((row['power_play_saves'] / row['power_play_shots_against']) * 100, 2)

# Calculate the recalculated power play save percentage
df_clean_game_goalie_stats['recalculated_power_play_save_percentage'] = df_clean_game_goalie_stats.apply(calculate_power_play_save_percentage, axis=1)

# Step 2: Integrity Check - Compare stored and recalculated power_play_save_percentage
def check_power_play_integrity(row):
    if row['power_play_shots_against'] == 0:
        return 'Correct' if pd.isnull(row['power_play_save_percentage']) else 'Incorrect'
    return 'Correct' if round(row['power_play_save_percentage'], 2) == row['recalculated_power_play_save_percentage'] else 'Incorrect'

df_clean_game_goalie_stats['power_play_integrity_check'] = df_clean_game_goalie_stats.apply(check_power_play_integrity, axis=1)

# Step 3: Handle power play save percentage status
def check_power_play_status(row):
    if row['power_play_shots_against'] == 0:
        return 'No shots against'
    elif pd.isnull(row['power_play_save_percentage']):
        return 'No data'
    else:
        return 'Data available'

df_clean_game_goalie_stats['power_play_status'] = df_clean_game_goalie_stats.apply(check_power_play_status, axis=1)

# Step 4: Identify rows where power_play_save_percentage is NULL
null_power_play_save_percentage_rows = df_clean_game_goalie_stats[df_clean_game_goalie_stats['power_play_save_percentage'].isnull()]

# Results
print(df_clean_game_goalie_stats[['game_id', 'player_id', 'team_id', 'power_play_shots_against', 'power_play_saves', 'power_play_save_percentage', 'recalculated_power_play_save_percentage', 'power_play_integrity_check', 'power_play_status']])
print(f"\nRows with NULL power_play_save_percentage: \n{null_power_play_save_percentage_rows}")

# Count the number of rows where power_play_integrity_check is not correct
incorrect_power_play_integrity_count = df_clean_game_goalie_stats[df_clean_game_goalie_stats['power_play_integrity_check'] == 'Incorrect'].shape[0]

# Output the result
print(f"Number of rows with power play integrity check not correct: {incorrect_power_play_integrity_count}")


          game_id  player_id  team_id  power_play_shots_against  \
0      2016020045    8473607        4                         3   
1      2016020045    8473461        4                         1   
2      2016020045    8470645       16                         4   
3      2017020812    8468011       24                         2   
4      2017020812    8475215        7                         5   
...           ...        ...      ...                       ...   
56651  2018030416    8476412       19                         4   
56652  2018030417    8476412       19                         3   
56653  2018030417    8471695        6                         0   
56654  2018030417    8476412       19                         3   
56655  2018030417    8471695        6                         0   

       power_play_saves  power_play_save_percentage  \
0                     1                   33.333333   
1                     1                  100.000000   
2                     2       

In [91]:
# Check integrity of even_strength_save_percentage column
# Step 1: Handle even_strength_save_percentage NULL values
def calculate_even_strength_save_percentage(row):
    if row['even_shots_against'] == 0:
        return None  # or you can return 100 if you want to treat 0 shots as perfect
    return round((row['even_saves'] / row['even_shots_against']) * 100, 2)

# Calculate the recalculated even strength save percentage
df_clean_game_goalie_stats['recalculated_even_strength_save_percentage'] = df_clean_game_goalie_stats.apply(calculate_even_strength_save_percentage, axis=1)

# Step 2: Integrity Check - Compare stored and recalculated even_strength_save_percentage
def check_even_strength_integrity(row):
    if row['even_shots_against'] == 0:
        return 'Correct' if pd.isnull(row['even_strength_save_percentage']) else 'Incorrect'
    return 'Correct' if round(row['even_strength_save_percentage'], 2) == row['recalculated_even_strength_save_percentage'] else 'Incorrect'

df_clean_game_goalie_stats['even_strength_integrity_check'] = df_clean_game_goalie_stats.apply(check_even_strength_integrity, axis=1)

# Step 3: Handle even strength save percentage status
def check_even_strength_status(row):
    if row['even_shots_against'] == 0:
        return 'No shots against'
    elif pd.isnull(row['even_strength_save_percentage']):
        return 'No data'
    else:
        return 'Data available'

df_clean_game_goalie_stats['even_strength_status'] = df_clean_game_goalie_stats.apply(check_even_strength_status, axis=1)

# Step 4: Identify rows where even_strength_save_percentage is NULL
null_even_strength_save_percentage_rows = df_clean_game_goalie_stats[df_clean_game_goalie_stats['even_strength_save_percentage'].isnull()]

# Results
print(df_clean_game_goalie_stats[['game_id', 'player_id', 'team_id', 'even_shots_against', 'even_saves', 'even_strength_save_percentage', 'recalculated_even_strength_save_percentage', 'even_strength_integrity_check', 'even_strength_status']])
print(f"\nRows with NULL even_strength_save_percentage: \n{null_even_strength_save_percentage_rows}")

# Count the number of rows where even_strength_integrity_check is not correct
incorrect_even_strength_integrity_count = df_clean_game_goalie_stats[df_clean_game_goalie_stats['even_strength_integrity_check'] == 'Incorrect'].shape[0]

# Output the result
print(f"Number of rows with even strength integrity check not correct: {incorrect_even_strength_integrity_count}")

          game_id  player_id  team_id  even_shots_against  even_saves  \
0      2016020045    8473607        4                  13          11   
1      2016020045    8473461        4                  10           8   
2      2016020045    8470645       16                  23          21   
3      2017020812    8468011       24                  28          27   
4      2017020812    8475215        7                  27          24   
...           ...        ...      ...                 ...         ...   
56651  2018030416    8476412       19                  26          23   
56652  2018030417    8476412       19                  30          29   
56653  2018030417    8471695        6                  20          16   
56654  2018030417    8476412       19                  30          29   
56655  2018030417    8471695        6                  20          16   

       even_strength_save_percentage  \
0                          84.615385   
1                          80.000000   
2  

In [92]:
print(f"Number of rows with save integrity check not correct: {incorrect_integrity_count}")
print(f"Number of rows with power play integrity check not correct: {incorrect_power_play_integrity_count}")
print(f"Number of rows with even strength integrity check not correct: {incorrect_even_strength_integrity_count}")

Number of rows with save integrity check not correct: 0
Number of rows with power play integrity check not correct: 0
Number of rows with even strength integrity check not correct: 0


In [97]:
# Round up the save_percentage, power_play_save_percentage, and even_strength_save_percentage to 2 decimal places
df_clean_game_goalie_stats['save_percentage'] = df_clean_game_goalie_stats['save_percentage'].apply(lambda x: round(x, 2) if pd.notnull(x) else x)
df_clean_game_goalie_stats['power_play_save_percentage'] = df_clean_game_goalie_stats['power_play_save_percentage'].apply(lambda x: round(x, 2) if pd.notnull(x) else x)
df_clean_game_goalie_stats['even_strength_save_percentage'] = df_clean_game_goalie_stats['even_strength_save_percentage'].apply(lambda x: round(x, 2) if pd.notnull(x) else x)

In [98]:
# Drop the previously added specified columns for purpose of calculation, from the DataFrame
columns_to_drop = [
    'recalculated_save_percentage',
    'integrity_check',
    'power_play_status',
    'recalculated_power_play_save_percentage',
    'power_play_integrity_check',
    'recalculated_even_strength_save_percentage',
    'even_strength_integrity_check',
    'even_strength_status'
]
df_clean_game_goalie_stats = df_clean_game_goalie_stats.drop(columns=columns_to_drop)

# Display the updated dataframe
df_clean_game_goalie_stats.info()

<class 'pandas.core.frame.DataFrame'>
Index: 56618 entries, 0 to 56655
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   game_id                        56618 non-null  int64  
 1   player_id                      56618 non-null  int64  
 2   team_id                        56618 non-null  int64  
 3   time_on_ice                    56618 non-null  int64  
 4   assists                        56618 non-null  int64  
 5   goals                          56618 non-null  int64  
 6   penalty_minutes                56618 non-null  int64  
 7   shots                          56618 non-null  int64  
 8   saves                          56618 non-null  int64  
 9   power_play_saves               56618 non-null  int64  
 10  short_handed_saves             56618 non-null  int64  
 11  even_saves                     56618 non-null  int64  
 12  short_handed_shots_against     56618 non-null  int6

In [101]:
df_clean_game_goalie_stats.head(10)

,game_id,player_id,team_id,time_on_ice,assists,goals,penalty_minutes,shots,saves,power_play_saves,short_handed_saves,even_saves,short_handed_shots_against,even_shots_against,power_play_shots_against,decision,save_percentage,power_play_save_percentage,even_strength_save_percentage
0,2016020045,8473607,4,1504,0,0,0,16,12,1,0,11,0,13,3,L,75.00,33.33,84.62
1,2016020045,8473461,4,2011,0,0,0,11,9,1,0,8,0,10,1,L,81.82,100.00,80.00
2,2016020045,8470645,16,3598,0,0,0,27,23,2,0,21,0,23,4,W,85.19,50.00,91.30
3,2017020812,8468011,24,3696,0,0,0,33,30,1,2,27,3,28,2,W,90.91,50.00,96.43
4,2017020812,8475215,7,3539,0,0,0,33,29,4,1,24,1,27,5,L,87.88,80.00,88.89
5,2015020314,8473575,21,3600,0,0,0,21,20,3,1,16,1,17,3,W,95.24,100.00,94.12
6,2015020314,8474636,52,3520,0,0,0,28,25,4,0,21,0,23,5,L,89.29,80.00,91.30
7,2015020849,8471715,52,3475,0,0,0,29,27,8,1,18,1,18,10,L,93.10,80.00,100.00
8,2015020849,8475663,12,3600,0,0,0,21,20,1,1,18,1,19,1,W,95.24,100.00,94.74
9,2017020586,8469608,20,3458,1,0,0,41,39,8,0,31,0,32,9,L,95.12,88.89,96.88


Check : Duplicates

In [104]:
# Count unique 'game_id' values
unique_ids = df_clean_game_goalie_stats['game_id'].nunique()

# Count total number of rows
total_rows = len(df_clean_game_goalie_stats)

# Display the results
print(f"Unique game_ids: {unique_ids}")
print(f"Total rows: {total_rows}")
print(f"There are {total_rows - unique_ids} rows with duplicates.")

# Results: Unique game_ids: 23721 / Total rows: 56656 / There are 32935 rows with duplicates.

Unique game_ids: 23721
Total rows: 56618
There are 32897 rows with duplicates.


In [106]:
# Result : 5493 duplicates
# Additional Duplicate Check for 'game_id','player_id', 'team_id' combinations
duplicates_id = df_clean_game_goalie_stats[df_clean_game_goalie_stats.duplicated(subset=['game_id','player_id', 'team_id'], keep=False)]

# Sort rows with duplicated results in ascending order by 'game_id','player_id', 'team_id'
duplicates_id_sorted = duplicates_id.sort_values(by=['game_id','player_id', 'team_id'], ascending=True)

# Count total number of rows with duplicates based on 'game_id','player_id', 'team_id'
total_duplicates= df_clean_game_goalie_stats.duplicated(subset=['game_id','player_id', 'team_id'], keep=False).sum()

# Count the unique combinations of 'game_id','player_id', 'team_id'
unique_combinations = df_clean_game_goalie_stats[['game_id','player_id', 'team_id']].drop_duplicates().shape[0]

# Count total number of rows
total_rows = len(df_clean_game_goalie_stats)

# Display the results
print(f"Unique 'game_id','player_id', 'team_id' combinations: {unique_combinations}")
print(f"Total rows: {total_rows}")
print(f"There are {total_rows - unique_combinations} rows with duplicates based on 'game_id','player_id', 'team_id'.")

# Display the sorted duplicates for further review
print("Sorted Duplicates based on 'game_id','player_id', 'team_id':")
print(duplicates_id_sorted[['game_id','player_id', 'team_id']])


Unique 'game_id','player_id', 'team_id' combinations: 51125
Total rows: 56618
There are 5493 rows with duplicates based on 'game_id','player_id', 'team_id'.
Sorted Duplicates based on 'game_id','player_id', 'team_id':
          game_id  player_id  team_id
50838  2018020001    8471679        8
50847  2018020001    8471679        8
50839  2018020001    8475883       10
50848  2018020001    8475883       10
50840  2018020002    8470860        6
...           ...        ...      ...
48958  2019040653    8475883       87
48945  2019040653    8476883       87
48957  2019040653    8476883       87
48947  2019040653    8479496       90
48959  2019040653    8479496       90

[10986 rows x 3 columns]


In [108]:
# Remove duplicates based on 'game_id','player_id', 'team_id'
df_clean_game_goalie_stats = df_clean_game_goalie_stats.drop_duplicates(subset=['game_id','player_id', 'team_id'], keep='first')

# Display the cleaned dataframe (first 10 rows as an example)
df_clean_game_goalie_stats.head(10)

,game_id,player_id,team_id,time_on_ice,assists,goals,penalty_minutes,shots,saves,power_play_saves,short_handed_saves,even_saves,short_handed_shots_against,even_shots_against,power_play_shots_against,decision,save_percentage,power_play_save_percentage,even_strength_save_percentage
0,2016020045,8473607,4,1504,0,0,0,16,12,1,0,11,0,13,3,L,75.00,33.33,84.62
1,2016020045,8473461,4,2011,0,0,0,11,9,1,0,8,0,10,1,L,81.82,100.00,80.00
2,2016020045,8470645,16,3598,0,0,0,27,23,2,0,21,0,23,4,W,85.19,50.00,91.30
3,2017020812,8468011,24,3696,0,0,0,33,30,1,2,27,3,28,2,W,90.91,50.00,96.43
4,2017020812,8475215,7,3539,0,0,0,33,29,4,1,24,1,27,5,L,87.88,80.00,88.89
5,2015020314,8473575,21,3600,0,0,0,21,20,3,1,16,1,17,3,W,95.24,100.00,94.12
6,2015020314,8474636,52,3520,0,0,0,28,25,4,0,21,0,23,5,L,89.29,80.00,91.30
7,2015020849,8471715,52,3475,0,0,0,29,27,8,1,18,1,18,10,L,93.10,80.00,100.00
8,2015020849,8475663,12,3600,0,0,0,21,20,1,1,18,1,19,1,W,95.24,100.00,94.74
9,2017020586,8469608,20,3458,1,0,0,41,39,8,0,31,0,32,9,L,95.12,88.89,96.88


In [110]:
# Final Duplicate Check: Check for duplicates based on 'game_id', 'player_id', and 'team_id'
duplicates_game_id = df_clean_game_goalie_stats[df_clean_game_goalie_stats.duplicated(subset=['game_id', 'player_id', 'team_id'], keep=False)]

if not duplicates_game_id.empty:
    # Sort the duplicated rows by 'game_id', 'player_id', and 'team_id'
    duplicates_game_id_sorted = duplicates_game_id.sort_values(by=['game_id', 'player_id', 'team_id'], ascending=True)

    # Print the sorted duplicated rows
    print("Sorted Duplicates based on 'game_id', 'player_id', 'team_id':")
    print(duplicates_game_id_sorted[['game_id', 'player_id', 'team_id']])
else:
    print("No duplicates found based on 'game_id', 'player_id', and 'team_id'. Data is clean!")


No duplicates found based on 'game_id', 'player_id', and 'team_id'. Data is clean!


Consistent Column Name - Python PEP Style Guide

In [113]:
df_clean_game_goalie_stats.info()

<class 'pandas.core.frame.DataFrame'>
Index: 51125 entries, 0 to 56653
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   game_id                        51125 non-null  int64  
 1   player_id                      51125 non-null  int64  
 2   team_id                        51125 non-null  int64  
 3   time_on_ice                    51125 non-null  int64  
 4   assists                        51125 non-null  int64  
 5   goals                          51125 non-null  int64  
 6   penalty_minutes                51125 non-null  int64  
 7   shots                          51125 non-null  int64  
 8   saves                          51125 non-null  int64  
 9   power_play_saves               51125 non-null  int64  
 10  short_handed_saves             51125 non-null  int64  
 11  even_saves                     51125 non-null  int64  
 12  short_handed_shots_against     51125 non-null  int6

Change Data Type

In [116]:
# Change data type syntax - df['column_name'] = df['column_name'].astype('desired_data_type')
df_clean_game_goalie_stats['game_id'] = df_clean_game_goalie_stats['game_id'].astype(str)
df_clean_game_goalie_stats['player_id'] = df_clean_game_goalie_stats['player_id'].astype(str)
df_clean_game_goalie_stats['team_id'] = df_clean_game_goalie_stats['team_id'].astype(str)
df_clean_game_goalie_stats.info()

<class 'pandas.core.frame.DataFrame'>
Index: 51125 entries, 0 to 56653
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   game_id                        51125 non-null  object 
 1   player_id                      51125 non-null  object 
 2   team_id                        51125 non-null  object 
 3   time_on_ice                    51125 non-null  int64  
 4   assists                        51125 non-null  int64  
 5   goals                          51125 non-null  int64  
 6   penalty_minutes                51125 non-null  int64  
 7   shots                          51125 non-null  int64  
 8   saves                          51125 non-null  int64  
 9   power_play_saves               51125 non-null  int64  
 10  short_handed_saves             51125 non-null  int64  
 11  even_saves                     51125 non-null  int64  
 12  short_handed_shots_against     51125 non-null  int6

In [118]:
df_clean_game_goalie_stats.to_csv(r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\clean\game_goalie_stats.csv", index=False)